In [1]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0)

## 1. Confirming CUDA is being used

In [2]:
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", GPU_NAME)

PyTorch: 2.14.0+cu130
CUDA: 13.0
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


## 2. Reproducibility and project paths

In [3]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

def find_project_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find project root.")

PROJECT_ROOT = find_project_root()
REPORT_DIR = PROJECT_ROOT / "reports" / "damage_hyperparameter_tuning"
MODEL_DIR = PROJECT_ROOT / "models" / "damage_hyperparameter_tuning"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development


## 3. Dataset YAML and weights

In [4]:
DATA_YAML = INTERIM_DIR / "cardd_damage_local.yaml"

BASELINE_CHECKPOINT = (
    PROJECT_ROOT
    / "models"
    / "damage_detection_benchmark"
    / "yolo11s_cardd_640px"
    / "weights"
    / "best.pt"
)

PRETRAINED_WEIGHT = PROJECT_ROOT / "notebooks" / "yolo11s.pt"

if not PRETRAINED_WEIGHT.exists():
    PRETRAINED_WEIGHT = Path("yolo11s.pt")

print("DATA_YAML:", DATA_YAML)
print("BASELINE_CHECKPOINT:", BASELINE_CHECKPOINT)
print("PRETRAINED_WEIGHT:", PRETRAINED_WEIGHT)

if not DATA_YAML.exists():
    raise FileNotFoundError("cardd_damage_local.yaml is missing.")
if not BASELINE_CHECKPOINT.exists():
    raise FileNotFoundError("Baseline YOLO11s best.pt is missing.")

DATA_YAML: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\data\interim\cardd_damage_local.yaml
BASELINE_CHECKPOINT: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\models\damage_detection_benchmark\yolo11s_cardd_640px\weights\best.pt
PRETRAINED_WEIGHT: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\notebooks\yolo11s.pt


## 4. Tuning configurations

In [5]:
EXPERIMENTS = [
    {
        "name": "640_longer",
        "imgsz": 640,
        "batch": 8,
        "epochs": 40,
        "patience": 10,
    },
    {
        "name": "800_highres",
        "imgsz": 800,
        "batch": 8,
        "epochs": 40,
        "patience": 10,
    },
    {
        "name": "960_highres",
        "imgsz": 960,
        "batch": 4,
        "epochs": 30,
        "patience": 8,
    },
]

EXPERIMENTS_TO_RUN = [
    "640_longer",
    "800_highres",
    "960_highres",
]

display(pd.DataFrame(EXPERIMENTS))

,name,imgsz,batch,epochs,patience
0,640_longer,640,8,40,10
1,800_highres,800,8,40,10
2,960_highres,960,4,30,8


## 5. Metric helpers

In [6]:
def metrics_to_dict(metrics):
    box = metrics.box
    return {
        "precision": float(box.mp),
        "recall": float(box.mr),
        "map50": float(box.map50),
        "map50_95": float(box.map),
        "inference_ms": float(metrics.speed.get("inference", np.nan)),
    }

def per_class_table(metrics, experiment_name, split):
    box = metrics.box

    p = np.asarray(getattr(box, "p", []))
    r = np.asarray(getattr(box, "r", []))
    ap50 = np.asarray(getattr(box, "ap50", []))
    maps = np.asarray(getattr(box, "maps", []))

    names = [
        "dent",
        "scratch",
        "crack",
        "glass shatter",
        "lamp broken",
        "tire flat",
    ]

    rows = []

    for i, class_name in enumerate(names):
        rows.append({
            "experiment": experiment_name,
            "split": split,
            "class_id": i,
            "damage_class": class_name,
            "precision": float(p[i]) if i < len(p) else np.nan,
            "recall": float(r[i]) if i < len(r) else np.nan,
            "ap50": float(ap50[i]) if i < len(ap50) else np.nan,
            "ap50_95": float(maps[i]) if i < len(maps) else np.nan,
        })

    return pd.DataFrame(rows)

## 6. Re-evaluating the baseline on validation

In [7]:
baseline_model = YOLO(str(BASELINE_CHECKPOINT))

baseline_val = baseline_model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=640,
    batch=8,
    workers=0,
    device=DEVICE,
    plots=False,
    verbose=False,
)

baseline_metrics = metrics_to_dict(baseline_val)

baseline_row = {
    "experiment": "baseline_640_30ep",
    "checkpoint": str(BASELINE_CHECKPOINT),
    "imgsz": 640,
    "batch": 8,
    "max_epochs": 30,
    **{f"val_{k}": v for k, v in baseline_metrics.items()},
}

display(pd.DataFrame([baseline_row]))

Ultralytics 8.4.171  Python-3.11.4 torch-2.14.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8123MiB)
YOLO11s summary (fused): 100 layers, 9,415,122 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access  (ping: 1.10.1 ms, read: 70.024.9 MB/s, size: 739.5 KB)
val: Scanning G:\.shortcut-targets-by-id\1OOvt005TduvEA8h6_snfuyQtbQAjo-3J\capstone-vehicle-metadata\data\cardd_kaggle\extracted\val\labels... 810 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 810/810 172.8it/s 4.7s0.1s
WARNING val: Failed to save cache to G:\.shortcut-targets-by-id\1OOvt005TduvEA8h6_snfuyQtbQAjo-3J\capstone-vehicle-metadata\data\cardd_kaggle\extracted\val\labels.cache: [Errno 13] Permission denied: 'G:\\.shortcut-targets-by-id\\1OOvt005TduvEA8h6_snfuyQtbQAjo-3J\\capstone-vehicle-metadata\\data\\cardd_kaggle\\extracted\\val\\labels.cache'
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 102/102 3.6it/s 28.4s0.3s
                   all      

,experiment,checkpoint,imgsz,batch,max_epochs,val_precision,val_recall,val_map50,val_map50_95,val_inference_ms
0,baseline_640_30ep,C:\Users\elesh\Academics\UTS\36127InnovationLa...,640,8,30,0.732367,0.688579,0.717842,0.571597,5.279931


## 7. Training the new configurations

In [8]:
all_results = [baseline_row]

per_class_results = [
    per_class_table(
        baseline_val,
        "baseline_640_30ep",
        "val",
    )
]

for config in EXPERIMENTS:
    if config["name"] not in EXPERIMENTS_TO_RUN:
        continue

    print("\n" + "=" * 90)
    print("EXPERIMENT:", config["name"])
    print(config)
    print("=" * 90)

    model = YOLO(str(PRETRAINED_WEIGHT))

    train_result = model.train(
        data=str(DATA_YAML),
        project=str(MODEL_DIR),
        name=config["name"],
        exist_ok=True,
        imgsz=config["imgsz"],
        batch=config["batch"],
        epochs=config["epochs"],
        patience=config["patience"],
        workers=0,
        device=DEVICE,
        seed=SEED,
        deterministic=True,
        amp=True,
        pretrained=True,
        cache=False,
        plots=True,
        verbose=True,
    )

    best_checkpoint = Path(train_result.save_dir) / "weights" / "best.pt"

    best_model = YOLO(str(best_checkpoint))

    val_metrics = best_model.val(
        data=str(DATA_YAML),
        split="val",
        imgsz=config["imgsz"],
        batch=config["batch"],
        workers=0,
        device=DEVICE,
        plots=False,
        verbose=False,
    )

    summary = metrics_to_dict(val_metrics)

    row = {
        "experiment": config["name"],
        "checkpoint": str(best_checkpoint),
        "imgsz": config["imgsz"],
        "batch": config["batch"],
        "max_epochs": config["epochs"],
        **{f"val_{k}": v for k, v in summary.items()},
    }

    all_results.append(row)

    per_class_results.append(
        per_class_table(
            val_metrics,
            config["name"],
            "val",
        )
    )

    print("VALIDATION RESULT:", row)

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


EXPERIMENT: 640_longer
{'name': '640_longer', 'imgsz': 640, 'batch': 8, 'epochs': 40, 'patience': 10}
New https://pypi.org/project/ultralytics/8.4.173 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.171  Python-3.11.4 torch-2.14.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8123MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\data\interim\cardd_damage_local.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, forma

## 8. Selecting the best using validation only

In [9]:
results_df = (
    pd.DataFrame(all_results)
    .sort_values(
        ["val_map50_95", "val_map50"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

display(results_df)

results_df.to_csv(
    REPORT_DIR / "damage_tuning_validation_results.csv",
    index=False,
)

per_class_df = pd.concat(
    per_class_results,
    ignore_index=True,
)

per_class_df.to_csv(
    REPORT_DIR / "damage_tuning_validation_per_class.csv",
    index=False,
)

selected = results_df.iloc[0]

print("\nSelected configuration:")
print(selected)

pd.DataFrame([selected]).to_csv(
    REPORT_DIR / "selected_damage_tuning_config.csv",
    index=False,
)

,experiment,checkpoint,imgsz,batch,max_epochs,val_precision,val_recall,val_map50,val_map50_95,val_inference_ms
0,640_longer,C:\Users\elesh\Academics\UTS\36127InnovationLa...,640,8,40,0.752430,0.695485,0.732004,0.580046,4.232676
1,800_highres,C:\Users\elesh\Academics\UTS\36127InnovationLa...,800,8,40,0.743064,0.674940,0.724229,0.575808,7.632240
2,baseline_640_30ep,C:\Users\elesh\Academics\UTS\36127InnovationLa...,640,8,30,0.732367,0.688579,0.717842,0.571597,5.279931
3,960_highres,C:\Users\elesh\Academics\UTS\36127InnovationLa...,960,4,30,0.723702,0.659723,0.707282,0.546572,9.955680



Selected configuration:
experiment                                                 640_longer
checkpoint          C:\Users\elesh\Academics\UTS\36127InnovationLa...
imgsz                                                             640
batch                                                               8
max_epochs                                                         40
val_precision                                                 0.75243
val_recall                                                   0.695485
val_map50                                                    0.732004
val_map50_95                                                 0.580046
val_inference_ms                                             4.232676
Name: 0, dtype: object


## 9. Comparing crack, dent, and scratch

In [10]:
focus_classes = ["crack", "dent", "scratch"]

focus_table = (
    per_class_df[
        per_class_df["damage_class"].isin(focus_classes)
    ]
    .pivot(
        index="damage_class",
        columns="experiment",
        values="ap50_95",
    )
)

display(focus_table)

focus_table.to_csv(
    REPORT_DIR / "surface_damage_ap50_95_comparison.csv"
)

experiment,640_longer,800_highres,960_highres,baseline_640_30ep
damage_class,,,,
crack,0.199844,0.228231,0.188200,0.180076
dent,0.323105,0.314593,0.295901,0.312587
scratch,0.331363,0.311519,0.288872,0.326634


## 10. Final test evaluation

In [11]:
SELECTED_CHECKPOINT = Path(selected["checkpoint"])
SELECTED_EXPERIMENT = str(selected["experiment"])
SELECTED_IMGSZ = int(selected["imgsz"])
SELECTED_BATCH = int(selected["batch"])

final_model = YOLO(str(SELECTED_CHECKPOINT))

test_metrics = final_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=SELECTED_IMGSZ,
    batch=SELECTED_BATCH,
    workers=0,
    device=DEVICE,
    plots=True,
    verbose=False,
)

test_summary = metrics_to_dict(test_metrics)

print("FINAL TEST RESULT")
print(test_summary)

pd.DataFrame([{
    "experiment": SELECTED_EXPERIMENT,
    "checkpoint": str(SELECTED_CHECKPOINT),
    **{f"test_{k}": v for k, v in test_summary.items()},
}]).to_csv(
    REPORT_DIR / "selected_damage_tuning_test_result.csv",
    index=False,
)

final_test_per_class = per_class_table(
    test_metrics,
    SELECTED_EXPERIMENT,
    "test",
)

display(final_test_per_class)

final_test_per_class.to_csv(
    REPORT_DIR / "selected_damage_tuning_test_per_class.csv",
    index=False,
)

Ultralytics 8.4.171  Python-3.11.4 torch-2.14.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8123MiB)
YOLO11s summary (fused): 100 layers, 9,415,122 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access  (ping: 0.70.1 ms, read: 64.213.1 MB/s, size: 735.3 KB)
val: Scanning G:\.shortcut-targets-by-id\1OOvt005TduvEA8h6_snfuyQtbQAjo-3J\capstone-vehicle-metadata\data\cardd_kaggle\extracted\test\labels... 374 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 374/374 89.9it/s 4.2s0.1ss
WARNING val: Failed to save cache to G:\.shortcut-targets-by-id\1OOvt005TduvEA8h6_snfuyQtbQAjo-3J\capstone-vehicle-metadata\data\cardd_kaggle\extracted\test\labels.cache: [Errno 13] Permission denied: 'G:\\.shortcut-targets-by-id\\1OOvt005TduvEA8h6_snfuyQtbQAjo-3J\\capstone-vehicle-metadata\\data\\cardd_kaggle\\extracted\\test\\labels.cache'
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 47/47 3.4it/s 13.9s0.3s
                   all     

,experiment,split,class_id,damage_class,precision,recall,ap50,ap50_95
0,640_longer,test,0,dent,0.667604,0.542373,0.625580,0.371602
1,640_longer,test,1,scratch,0.664283,0.579805,0.643381,0.363774
2,640_longer,test,2,crack,0.573902,0.414286,0.437500,0.265889
3,640_longer,test,3,glass shatter,0.896474,0.985915,0.976958,0.904556
4,640_longer,test,4,lamp broken,0.826446,0.753623,0.852805,0.709977
5,640_longer,test,5,tire flat,0.959337,0.843750,0.913517,0.878029
